# MP2 Parts 2-4 - Visualization, gaps and interpretation (netid: gkhot)

In [1]:
import os, re, time
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests

netid = 'gkhot'
df = pd.read_csv(f'{netid}_project_summary.csv', sep=';')
df.columns = ['project', 'commit', 'author', 'time', 'message']
df['date'] = pd.to_datetime(df['time'], unit='s')
df['Month'] = df['date'].dt.strftime('%Y-%m')
projects = sorted(df['project'].unique())
part1 = pd.read_csv('cache/part1_stats.csv')
df.groupby('project').size()

Matplotlib is building the font cache; this may take a moment.


/Users/gururajkhot/Mini-Project-2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


project
alanwilter_acpype                 665
aura-healthcare_hrv-analysis      168
dakep_pense-rpkg                  728
declaredesign_estimatr           1297
igsio_openigtlinkio               592
ipeagit_geobr                    4695
juliaopt_cplex.jl                 567
rrrlw_ripserr                     314
rwth-i6_returnn                 12167
sheffieldml_gpy                  6322
dtype: int64

## Part 2: monthly time series
One row per month from the first to the last month with a commit; months without commits are filled with 0.

In [2]:
def monthly_series(prj):
    d = df[df['project'] == prj]
    counts = d.groupby('Month').size()
    months = pd.period_range(d['date'].min().to_period('M'), d['date'].max().to_period('M'), freq='M')
    ts = pd.DataFrame({'Month': months.strftime('%Y-%m')})
    ts['#Commits'] = ts['Month'].map(counts).fillna(0).astype(int)
    return ts

series = {}
for prj in projects:
    ts = monthly_series(prj)
    series[prj] = ts
    ts.to_csv(f'{netid}_commits_timeseries_{prj}.csv', sep=';', index=False)
    print(f'{prj}: {len(ts)} months, {ts["#Commits"].sum()} commits')

alanwilter_acpype: 88 months, 665 commits
aura-healthcare_hrv-analysis: 75 months, 168 commits
dakep_pense-rpkg: 103 months, 728 commits
declaredesign_estimatr: 95 months, 1297 commits
igsio_openigtlinkio: 152 months, 592 commits
ipeagit_geobr: 72 months, 4695 commits
juliaopt_cplex.jl: 136 months, 567 commits
rrrlw_ripserr: 81 months, 314 commits
rwth-i6_returnn: 133 months, 12167 commits
sheffieldml_gpy: 147 months, 6322 commits


In [3]:
for prj, ts in series.items():
    x = pd.to_datetime(ts['Month'])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(x, ts['#Commits'], marker='o', markersize=2.5, linestyle='-', linewidth=1)
    ax.set_xlabel('Time (YYYY-MM)')
    ax.set_ylabel('Number of Commits')
    ax.set_title(prj)
    ax.grid(True, alpha=0.5)
    span_years = len(ts) / 12
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=max(1, int(round(span_years / 2)))))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    fig.tight_layout()
    fig.savefig(f'{netid}_timeseries_{prj}.png', dpi=300)
    plt.close(fig)
print('saved', len(series), 'plots')

saved 10 plots


## Longest inactivity gap and number of gaps (>= 3 zero months)
Gap start/end are the first and last zero-commit months of the run (inclusive). If several runs tie for the longest, the earliest is used.

In [4]:
def zero_runs(ts):
    runs, start = [], None
    for i, c in enumerate(ts['#Commits']):
        if c == 0 and start is None:
            start = i
        if c != 0 and start is not None:
            runs.append((start, i - 1)); start = None
    return runs  # series always ends in a non-zero month

gap_rows = []
for prj, ts in series.items():
    runs = zero_runs(ts)
    if runs:
        s, e = max(runs, key=lambda r: (r[1] - r[0] + 1, -r[0]))
        gstart, gend, glen = ts['Month'][s], ts['Month'][e], e - s + 1
        after = int(ts['#Commits'][e + 1:].sum())
    else:
        gstart = gend = 'N/A'; glen = 0; after = int(ts['#Commits'].sum())
    gap_rows.append({'project_wocid': prj, 'LongestGapStart': gstart, 'LongestGapEnd': gend,
                     'LongestGapLength': glen, 'NumCommitsAfterLastGap': after,
                     'NumberOfGapsInTimeline': sum(1 for a, b in runs if b - a + 1 >= 3)})
gaps = pd.DataFrame(gap_rows)
gaps

,project_wocid,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,alanwilter_acpype,2019-04,2019-06,3,483,6
1,aura-healthcare_hrv-analysis,2021-05,2024-08,40,22,4
2,dakep_pense-rpkg,2016-09,2017-03,7,546,6
3,declaredesign_estimatr,2022-08,2023-06,11,23,5
4,igsio_openigtlinkio,2019-11,2020-09,11,20,11
5,ipeagit_geobr,2022-09,2022-12,4,260,1
6,juliaopt_cplex.jl,2024-06,2024-12,7,1,5
7,rrrlw_ripserr,2023-01,2024-09,21,2,6
8,rwth-i6_returnn,2014-07,2014-08,2,12066,0
9,sheffieldml_gpy,2022-05,2023-01,9,328,4


### Yearly totals (helps classify the activity pattern)

In [5]:
yearly = df.assign(Year=df['date'].dt.year).pivot_table(index='project', columns='Year',
                                                          values='commit', aggfunc='count', fill_value=0)
yearly

Year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
project,,,,,,,,,,,,,,,
alanwilter_acpype,0,0,0,0,0,0,65,105,54,50,297,34,45,13,2
aura-healthcare_hrv-analysis,0,0,0,0,0,0,0,117,9,5,15,0,0,22,0
dakep_pense-rpkg,0,0,0,0,0,182,198,22,18,81,84,73,39,31,0
declaredesign_estimatr,0,0,0,0,0,0,371,614,128,100,41,20,5,11,7
igsio_openigtlinkio,46,47,40,60,34,116,163,55,11,9,3,5,3,0,0
ipeagit_geobr,0,0,0,0,0,0,0,0,3631,484,239,81,112,144,4
juliaopt_cplex.jl,0,0,73,104,69,48,52,52,59,45,23,17,20,4,1
rrrlw_ripserr,0,0,0,0,0,0,0,15,8,208,75,6,0,2,0
rwth-i6_returnn,0,0,0,171,1662,1655,1443,1121,1427,806,783,868,1500,661,70


# Interpretation

Number of gaps = runs of at least 3 consecutive months with zero commits (see the table above).

- **alanwilter_acpype — irregular, 6 gaps.** 50-100 commits most years with a big spike in 2021 (~300), but it pauses for exactly 3 months six different times. None of the pauses is long, they look like a solo maintainer's breaks.
- **aura-healthcare_hrv-analysis — declining, 4 gaps.** Almost everything happened in 2018 (117 commits). After that it's single digits per year and then a 40-month gap (2021-05 to 2024-08) before a short burst of fixes in late 2024.
- **dakep_pense-rpkg — U-shaped, 6 gaps.** High in 2016-2017, very low in 2018-2019, then back up in 2020-2022 before slowing down again. The dip and the comeback are why I picked U-shaped.
- **declaredesign_estimatr — declining, 5 gaps.** 600+ commits in 2018 then lower every year; the gaps get longer near the end (11 months, then 9 months).
- **igsio_openigtlinkio — declining, 11 gaps.** Peaks in 2016-2017 and then only a handful of commits per year, so after 2019 most of the timeline is gaps.
- **ipeagit_geobr — declining, 1 gap.** A massive first year (3,600+ commits in 2019) followed by much lower but fairly stable activity (~100-150/year). Only one 4-month gap.
- **juliaopt_cplex.jl — declining, 5 gaps.** Steady for years (45-100 commits/year until 2020), then it goes down and all five gaps are after 2021.
- **rrrlw_ripserr — irregular, 6 gaps.** Almost nothing in 2018-2019, a sudden spike in 2020 (200+ commits), then long gaps of 16 and 21 months. No real trend, just bursts.
- **rwth-i6_returnn — steady, 0 gaps.** Between ~700 and ~1,700 commits every year since 2015 with no gap of 3+ months at all.
- **sheffieldml_gpy — declining, 4 gaps.** 2,000+ commits in 2013 dropping to 2 in 2022, with all four gaps clustered in 2021-2023. It does revive in 2023-2024 (~160/year), but the overall trajectory is down.

None of my projects looked seasonal/cyclical to me: the dips don't repeat at the same time each year.

## Part 3.1: commits before and after the longest gap
Last 10 commits before the gap start and the first 10 commits after the gap end.

In [6]:
gap_parts = []
for _, g in gaps.iterrows():
    prj = g['project_wocid']
    d = df[df['project'] == prj].sort_values('time')
    if g['LongestGapLength'] == 0:
        continue
    pre = d[d['Month'] < g['LongestGapStart']].tail(10).assign(prepost='pre')
    post = d[d['Month'] > g['LongestGapEnd']].head(10).assign(prepost='post')
    gap_parts += [pre, post]
gap_commits = pd.concat(gap_parts)[['prepost', 'project', 'commit', 'author', 'time', 'message']]
gap_commits.columns = ['prepost', 'project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
gap_commits.to_csv(f'{netid}_project_gap_commits.csv', sep=';', index=False)
gap_commits.groupby(['project_wocid', 'prepost']).size().unstack()

prepost,post,pre
project_wocid,,
alanwilter_acpype,10,10
aura-healthcare_hrv-analysis,10,10
dakep_pense-rpkg,10,10
declaredesign_estimatr,10,10
igsio_openigtlinkio,10,10
ipeagit_geobr,10,10
juliaopt_cplex.jl,1,10
rrrlw_ripserr,2,10
rwth-i6_returnn,10,10


### Theme labelling
A keyword pass gives a first label for every commit; I then read every message (printed below) and chose the top two themes per window by hand (recorded in the final table).

In [7]:
THEMES = [
    ('Automated bot contributions', r'dependabot|\[bot\]|pre-commit-ci|github-actions|renovate|auto-?format|black formatting|codecov'),
    ('Release', r'\brelease|\bbump(ed)? version|version bump|\bv?\d+\.\d+(\.\d+)?\b.*(release|tag)|^v?\d+\.\d+(\.\d+)?$|cran|changelog|news\.md|pypi|prepare for'),
    ('Security patches', r'security|vulnerab|\bcve\b'),
    ('Dependency updates', r'depend|requirement|bump|upgrade|update .*(version|package)|pin |numpy|scipy|setup\.py|pyproject|description file'),
    ('Documentation updates', r'doc|readme|vignette|typo|comment|example|tutorial|website|pkgdown|man/|roxygen'),
    ('Bug fixes', r'\bfix|bug|error|issue|crash|wrong|correct|broken|fail|problem|patch|hotfix|resolve'),
    ('Feature development', r'\badd|implement|feature|new |support|introduc|enhanc|improv|refactor|allow|option|extend'),
]
def theme(msg):
    m = str(msg).lower()
    if m.startswith('merge pull request') or m.startswith('merge branch'):
        m = m.split('\n', 2)[-1] if '\n' in m else m
    for name, pat in THEMES:
        if re.search(pat, m):
            return name
    return 'Other'

gap_commits['theme'] = gap_commits['commit message'].map(theme)
pd.set_option('display.max_colwidth', 120)
for (prj, pp), g in gap_commits.groupby(['project_wocid', 'prepost'], sort=False):
    print(f'=== {prj} [{pp}]', g['theme'].value_counts().head(3).to_dict())
    for _, r in g.iterrows():
        when = pd.to_datetime(r['time'], unit='s').strftime('%Y-%m-%d')
        print(f'  {when} {r["author"][:30]:30} | {r["theme"]:25} | {str(r["commit message"]).splitlines()[0][:90]}')

=== alanwilter_acpype [pre] {'Other': 4, 'Feature development': 3, 'Documentation updates': 2}
  2019-02-12 Alan Silva <3899850+alanwilter | Other                     | Removed ACS, it's in bitbucket now
  2019-03-04 Alan Silva <3899850+alanwilter | Documentation updates     | While parsing PRMTOP files, ignore non-existent attributes and %COMMENT
  2019-03-06 Alan <3899850+alanwilter@users | Documentation updates     | Typo
  2019-03-22 Alan Silva <3899850+alanwilter | Bug fixes                 | Fixed dict.keys list issues thanks to Vedat Durmaz
  2019-03-22 Alan Silva <3899850+alanwilter | Other                     | Default timeout is now 3h instead of 10h
  2019-03-22 Alan Silva <3899850+alanwilter | Other                     | Merge branch 'master' of https://github.com/alanwilter/acpype.git
  2019-03-22 Alan Silva <3899850+alanwilter | Other                     | Time stamp updated
  2019-03-29 lkagami <luciano.kagami@ufrgs. | Feature development       | Add Anaconda package fil

## Part 3.2: current status and recent commits (GitHub)
WoC stops in early 2025, so the 10 most recent commits are fetched from the GitHub API. Active = a commit between 2025-04-01 and 2025-09-30 (the README's 6-month rule as of 2025-09-30).

In [8]:
headers = {'Accept': 'application/vnd.github+json'}
if os.environ.get('GITHUB_TOKEN'):
    headers['Authorization'] = 'Bearer ' + os.environ['GITHUB_TOKEN']
recent_rows = []
for _, r in part1.iterrows():
    full = r['GH'].rstrip('/').split('github.com/')[1]
    commits = requests.get(f'https://api.github.com/repos/{full}/commits',
                           params={'per_page': 10}, headers=headers).json()
    for c in commits:
        recent_rows.append({'project_wocid': r['project_wocid'],
                            'date': c['commit']['committer']['date'][:10],
                            'author': c['commit']['author']['name'],
                            'message': c['commit']['message']})
    time.sleep(1)
recent = pd.DataFrame(recent_rows)
recent['theme'] = recent['message'].map(theme)
recent.to_csv('cache/recent_gh_commits.csv', index=False)
# status is defined as of 2025-09-30, so use the last commit on or before that date (from the git history)
import subprocess
def last_commit_by(prj, date='2025-09-30'):
    return subprocess.run(['git', '-C', f'../ghclones/{prj}.git', 'log', '--all', '--format=%cs',
                           f'--until={date}T23:59:59', '-1'], capture_output=True, text=True).stdout.strip()
status = part1[['project_wocid', 'lastGHCommitDate']].copy()
status['lastCommitBy2025-09-30'] = status['project_wocid'].map(last_commit_by)
status['Currentstatus'] = status['lastCommitBy2025-09-30'].map(lambda d: 'Active' if d >= '2025-04-01' else 'Inactive')
for prj, g in recent.groupby('project_wocid', sort=False):
    print(f'=== {prj}', g['theme'].value_counts().head(3).to_dict())
    for _, r in g.iterrows():
        print(f'  {r["date"]} {r["author"][:25]:25} | {r["theme"]:25} | {r["message"].splitlines()[0][:90]}')
status

=== alanwilter_acpype {'Dependency updates': 4, 'Automated bot contributions': 4, 'Bug fixes': 1}
  2026-09-24 Alan Silva                | Dependency updates        | Merge pull request #153 from alanwilter/dependabot/uv/typer-0.27.2
  2026-09-24 Alan Silva                | Dependency updates        | Merge pull request #152 from alanwilter/dependabot/uv/dev-dependencies-564d0085cf
  2026-09-24 Alan Silva                | Dependency updates        | Merge pull request #151 from alanwilter/dependabot/github_actions/actions-3a0b6790fd
  2026-09-24 dependabot[bot]           | Automated bot contributions | build(deps): bump typer from 0.27.1 to 0.27.2
  2026-09-24 dependabot[bot]           | Automated bot contributions | build(deps): bump ruff in the dev-dependencies group
  2026-09-24 dependabot[bot]           | Automated bot contributions | ci: bump astral-sh/setup-uv from 10.1.0 to 10.2.0 in the actions group
  2026-09-17 Alan Silva                | Dependency updates        | Merge pul

,project_wocid,lastGHCommitDate,lastCommitBy2025-09-30,Currentstatus
0,alanwilter_acpype,2026-09-24,2025-06-10,Active
1,aura-healthcare_hrv-analysis,2026-06-02,2024-11-11,Inactive
2,dakep_pense-rpkg,2026-01-27,2025-07-23,Active
3,declaredesign_estimatr,2026-09-21,2025-02-28,Inactive
4,igsio_openigtlinkio,2025-07-18,2025-07-18,Active
5,ipeagit_geobr,2026-09-27,2025-08-04,Active
6,juliaopt_cplex.jl,2026-07-24,2025-04-04,Active
7,rrrlw_ripserr,2026-10-01,2025-09-30,Active
8,rwth-i6_returnn,2026-10-07,2025-09-22,Active
9,sheffieldml_gpy,2026-10-07,2025-08-15,Active


## Part 3/4: manual judgments
Themes, gap reasons and recovery were decided by reading the commit messages above plus the projects' GitHub issues, PRs, releases and READMEs around the gap. Full reasoning is in `gkhot_reflection_<WOCProjectID>.md`.

In [9]:
manual = {
    'alanwilter_acpype': dict(
        ActivityPattern='irregular',
        BeforeThemes='Bug fixes:Other', AfterThemes='Documentation updates:Release',
        HypothesizedGapReason='Short break by the solo maintainer (Alan Silva); last pre-gap commits were PRMTOP parsing fixes and Anaconda packaging files (2019-03), nothing indicating trouble and no unanswered issues',
        HasRecovered='Yes',
        WhyRecovered='Maintainer returned to clean up: "Refactoring anaconda", updated citations/tutorial and "Release 2019.07.02" (2019-07-02)',
        WhoRecovered='Same person: Alan Silva (original author)',
        RecentThemes='Automated bot contributions:Dependency updates',
        Notes='Irregular project with six 3-month pauses and a 2021 spike of ~300 commits. The longest gap is only 3 months and reads as a solo maintainer taking a break before a July 2019 release. Recent activity is mostly Dependabot bumps merged by the author.'),
    'aura-healthcare_hrv-analysis': dict(
        ActivityPattern='declining',
        BeforeThemes='Documentation updates:Release', AfterThemes='Bug fixes:Dependency updates',
        HypothesizedGapReason='Project considered finished after the JORS paper submission and PyPI release (PR #27, 2021-02); afterwards issues went unanswered, incl. 3 reports that LombScargle could not be imported from newer astropy (#41-#43) and an unmerged fix PR #40',
        HasRecovered='Yes (briefly)',
        WhyRecovered='Users could no longer import the package; Sept 2024 PR #44 "fix: working LombScargle with recent astropy versions" plus new CI/publish workflow',
        WhoRecovered='New people from the Aura-healthcare org (Alexis Comte, Laura Dumont), not the original author Robin Champseix',
        RecentThemes='Automated bot contributions:Dependency updates',
        Notes='Almost all work happened in 2018; the 40-month gap (2021-05 to 2024-08) followed the JORS publication. A breaking astropy change with unanswered issues prompted new org members to fix it in 2024, but activity stopped again after Nov 2024 so it is Inactive as of 2025-09-30.'),
    'dakep_pense-rpkg': dict(
        ActivityPattern='U-shaped',
        BeforeThemes='Bug fixes:Release', AfterThemes='Feature development:Bug fixes',
        HypothesizedGapReason='One-developer research package (davidkep); gap follows hotfix releases 0.5.5/0.5.6 and has no issues or messages explaining it, most likely the developer was busy with other (academic) work',
        HasRecovered='Yes',
        WhyRecovered='Developer resumed work on new features (lambda grid options, decreasing lambda sequence) and bug fixes in April 2017',
        WhoRecovered='Same person: davidkep (David Kepplinger)',
        RecentThemes='Bug fixes:Feature development',
        Notes='Busy 2016-2017, almost idle 2018-2019, active again 2020-2022. The 7-month 2016-17 gap and the later ones follow one developer\'s schedule. It came back each time through the same author.'),
    'declaredesign_estimatr': dict(
        ActivityPattern='declining',
        BeforeThemes='Release:Dependency updates', AfterThemes='Release:Documentation updates',
        HypothesizedGapReason='Mature package went quiet right after the June 2022 CRAN submission ("submitted to cran"); issues #395-#405 got few replies and PR #394 "New version 1.0.0" stayed open the whole gap',
        HasRecovered='Partially',
        WhyRecovered='Maintainer merged the long-open PR #394 and updated docs (2023-07-28) to finish the 1.0.0 release; later only small fixes',
        WhoRecovered='Same people: Graeme Blair, later Alexander Coppock',
        RecentThemes='Release:Dependency updates',
        Notes='Heavy development in 2017-2018 then steady decline. The 11-month gap began right after a CRAN submission and ended when the same maintainer merged the 1.0.0 PR. With no commits after Feb 2025 it is Inactive as of 2025-09-30.'),
    'igsio_openigtlinkio': dict(
        ActivityPattern='declining',
        BeforeThemes='Feature development:Bug fixes', AfterThemes='Dependency updates:Bug fixes',
        HypothesizedGapReason='Library was stable after the 2019 TDATA/device additions so nothing needed changing; no abandonment signals in commits or issues',
        HasRecovered='Yes',
        WhyRecovered='The VTK 9 release broke compilation: "COMP: Enabling compilation with vtk >= 8.9.0" (#82), issue #83 "Build error with VTK8", "fixes to build with VTK 9.0.1 on Mac" (#85)',
        WhoRecovered='Mix: existing contributors Adam Rankin and Kyle Sunderland plus new ones (Simon Drouin, Ole Vegard Solberg, Laura Connolly)',
        RecentThemes='Dependency updates:Bug fixes',
        Notes='Peak in 2016-2017, then a few commits per year with 11 gaps of 3+ months. The 11-month gap ended when VTK 9 forced compilation fixes. Activity since is mostly keeping it building with new dependencies (CMake, Qt 6).'),
    'ipeagit_geobr': dict(
        ActivityPattern='declining',
        BeforeThemes='Release:Automated bot contributions', AfterThemes='Feature development:Automated bot contributions',
        HypothesizedGapReason='Normal break after the v1.7.0 release (Aug 2022) over the end-of-year holidays; no problem signals in issues',
        HasRecovered='Yes',
        WhyRecovered='Lead developer resumed feature work: new read_capitals() function (2023-01-29)',
        WhoRecovered='Same person: Rafael H. M. Pereira (lead maintainer), plus small outside PR from Vitor Sousa',
        RecentThemes='Release:Bug fixes',
        Notes='Huge first year (3,600+ commits in 2019) then lower but stable activity. Only one gap of 3+ months, a 4-month post-release/holiday pause. Still very active with R, Python and QGIS releases.'),
    'juliaopt_cplex.jl': dict(
        ActivityPattern='declining',
        BeforeThemes='Release:Documentation updates', AfterThemes='Documentation updates',
        HypothesizedGapReason='Mature solver wrapper in maintenance mode with one maintainer (Oscar Dowson) who maintains many JuMP wrappers; changes only needed when CPLEX/MOI change, and there were no unanswered bug reports',
        HasRecovered='Yes (maintenance mode)',
        WhyRecovered='Routine maintenance: README badge fix (#450, 2025-01), then v1.1.1 release (2025-04) and CPLEX 22.2 support (2026)',
        WhoRecovered='Same person: Oscar Dowson',
        RecentThemes='Automated bot contributions:Release',
        Notes='Steady 2013-2020, then declining with five gaps after 2021. The 7-month 2024 gap is a finished wrapper waiting for upstream changes. Only one commit after it in the WoC window (so only one After theme), but it has continued with releases.'),
    'rrrlw_ripserr': dict(
        ActivityPattern='irregular',
        BeforeThemes='Documentation updates:Feature development', AfterThemes='Feature development:Documentation updates',
        HypothesizedGapReason='Lack of maintainer time: PR #34 (opened 2022-12-08) sat unmerged until 2024-10-09 and mid-2022 issues (#30, #31) were not resolved',
        HasRecovered='Yes',
        WhyRecovered='Maintainer merged the pending PR #34 and updated documentation (Oct 2024); repo then moved to the tdaverse org with heavy 2025 activity',
        WhoRecovered='Same person: Cory Brunson (contributor before the gap)',
        RecentThemes='Bug fixes:Feature development',
        Notes='Irregular history with a 2020 spike and two very long gaps (16 and 21 months). The longest gap is a PR waiting almost two years for review. It recovered through the same contributor and is very active in 2025.'),
    'rwth-i6_returnn': dict(
        ActivityPattern='steady',
        BeforeThemes='Feature development:Bug fixes', AfterThemes='Feature development:Release',
        HypothesizedGapReason='N/A active Project',
        HasRecovered='N/A',
        WhyRecovered='N/A (never inactive; the only 2-month pause was in 2014 before the open-source release, "towards open source release")',
        WhoRecovered='N/A',
        RecentThemes='Feature development:Bug fixes',
        Notes='Largest project (12k+ commits) and never inactive for 3+ months. Roughly 700-1,700 commits every year since 2015 from a research group led in commits by Albert Zeyer. Still actively developing the PyTorch backend.'),
    'sheffieldml_gpy': dict(
        ActivityPattern='declining',
        BeforeThemes='Bug fixes:Feature development', AfterThemes='Dependency updates:Release',
        HypothesizedGapReason='Original developers had moved on and nobody was maintaining it; meanwhile GPy broke for users on new Python/numpy (issues #998 install error, #1004 incompatible with numpy >= 1.24, 10 comments)',
        HasRecovered='Yes',
        WhyRecovered='Users needed it to work on modern Python: PR #1011 "Fix issues encountered in modern python versions", pyproject.toml packaging, new release (2023)',
        WhoRecovered='Mostly a new person: Martin Bubel (304 commits 2023-2025), plus KOLANICH; Zhenwen Dai/Neil Lawrence from the original team for releases and reviews',
        RecentThemes='Bug fixes:Other',
        Notes='Huge 2013-2015 activity that declined to 2 commits in 2022, with four gaps clustered in 2021-2023. Breakage on new numpy/Python led a new contributor, Martin Bubel, to revive it in 2023. It is active again, with the original lead merging fixes.'),
}

## Final `gkhot_project_stats.csv`

In [10]:
stats = part1.rename(columns={'project_wocid': 'Project', 'from': 'from', 'to': 'to'})
stats = stats[['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate']]
stats = stats.merge(gaps.rename(columns={'project_wocid': 'Project',
                                         'LongestGapStart': 'LongestGapStart (YYYY-MM)',
                                         'LongestGapEnd': 'LongestGapEnd (YYYY-MM)',
                                         'LongestGapLength': 'LongestGapLength (months)'}), on='Project')
stats = stats.merge(status.rename(columns={'project_wocid': 'Project'})[['Project', 'Currentstatus']], on='Project')
manual_df = pd.DataFrame.from_dict(manual, orient='index').rename_axis('Project').reset_index()
stats = stats.merge(manual_df, on='Project')
cols = ['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate',
        'LongestGapStart (YYYY-MM)', 'LongestGapEnd (YYYY-MM)', 'LongestGapLength (months)',
        'NumCommitsAfterLastGap', 'ActivityPattern', 'NumberOfGapsInTimeline',
        'BeforeThemes', 'AfterThemes',
        'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'WhoRecovered',
        'Currentstatus', 'RecentThemes', 'Notes']
stats = stats[cols]
stats.to_csv(f'{netid}_project_stats.csv', sep=';', index=False)
stats[cols[:16]]

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart (YYYY-MM),LongestGapEnd (YYYY-MM),LongestGapLength (months),NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline,BeforeThemes,AfterThemes
0,alanwilter_acpype,665,18,2017-10-20,2025-01-20,263,61,2026-09-24,2019-04,2019-06,3,483,irregular,6,Bug fixes:Other,Documentation updates:Release
1,aura-healthcare_hrv-analysis,168,13,2018-09-13,2024-11-11,460,111,2026-06-02,2021-05,2024-08,40,22,declining,4,Documentation updates:Release,Bug fixes:Dependency updates
2,dakep_pense-rpkg,728,12,2016-02-05,2024-08-10,5,2,2026-01-27,2016-09,2017-03,7,546,U-shaped,6,Bug fixes:Release,Feature development:Bug fixes
3,declaredesign_estimatr,1297,26,2017-03-24,2025-01-22,135,22,2026-09-21,2022-08,2023-06,11,23,declining,5,Release:Dependency updates,Release:Documentation updates
4,igsio_openigtlinkio,592,35,2011-04-06,2023-11-15,14,26,2025-07-18,2019-11,2020-09,11,20,declining,11,Feature development:Bug fixes,Dependency updates:Bug fixes
5,ipeagit_geobr,4695,46,2019-02-27,2025-01-21,959,131,2026-09-27,2022-09,2022-12,4,260,declining,1,Release:Automated bot contributions,Feature development:Automated bot contributions
6,juliaopt_cplex.jl,567,57,2013-10-12,2025-01-05,139,62,2026-07-24,2024-06,2024-12,7,1,declining,5,Release:Documentation updates,Documentation updates
7,rrrlw_ripserr,314,16,2018-02-15,2024-10-10,12,7,2026-10-01,2023-01,2024-09,21,2,irregular,6,Documentation updates:Feature development,Feature development:Documentation updates
8,rwth-i6_returnn,12167,159,2014-01-14,2025-01-31,377,136,2026-10-07,2014-07,2014-08,2,12066,steady,0,Feature development:Bug fixes,Feature development:Release
9,sheffieldml_gpy,6322,153,2012-11-28,2025-01-15,2169,571,2026-10-07,2022-05,2023-01,9,328,declining,4,Bug fixes:Feature development,Dependency updates:Release
